# CSC4120 Project - Party Together Problem

This notebook tests all implementations:
- **Question 2**: Input Generation (4 input files)
- **Question 3**: PTP Solver (insert/delete heuristic)
- **Question 4.1**: M-TSP Solver (Held-Karp DP)
- **Question 4.2**: PHP Solver (reduction to TSP)

In [ ]:
# Import all required modules
import networkx as nx
from student_utils import (
    input_file_to_instance, 
    analyze_solution, 
    is_valid_input,
    write_ptp_solution_to_out
)
from ptp_solver import ptp_solver
from php_from_tsp import php_solver_from_tsp
from mtsp_dp import mtsp_dp
import os

---
## PDF Example (Figure 1) - Verification

Test the example from the project description:
- **Graph**: 4 nodes (0, 1, 2, 3)
- **Homes**: H = {1, 2, 3}
- **Alpha**: 2/3 ≈ 0.66667
- **Expected**: Tour [0, 1, 0], everyone picked up at node 1
- **Expected Cost**: 10/3 ≈ 3.333

In [ ]:
print("=" * 60)
print("PDF EXAMPLE (Figure 1) - VERIFICATION")
print("=" * 60)

G, H, alpha = input_file_to_instance('inputs/example.in')
print(f"\nGraph: {G.number_of_nodes()} nodes")
print(f"Homes H: {H}")
print(f"Alpha: {alpha}")

print("\nEdges:")
for u, v, d in G.edges(data=True):
    if u < v:  # Only print each edge once
        print(f"  {u} -- {v}: weight={int(d['weight'])}")

print("\n--- Expected Solution (from PDF) ---")
print("Tour: [0, 1, 0]")
print("Pickup: {1: [1, 2, 3]}")
print("Expected Cost: 10/3 = 3.3333")

print("\n--- My Solver Output ---")
tour, pickups = ptp_solver(G, H, alpha)
print(f"Tour: {tour}")
print(f"Pickups: {pickups}")

is_valid, drive_cost, walk_cost = analyze_solution(G, H, alpha, tour, pickups)
total = drive_cost + walk_cost
print(f"\nDriving Cost: {drive_cost:.4f}")
print(f"Walking Cost: {walk_cost:.4f}")
print(f"Total Cost: {total:.4f}")

expected = 10/3
if is_valid and abs(total - expected) < 0.01:
    print(f"\n>>> PASS - Matches expected optimal solution! <<<")
else:
    print(f"\n>>> FAIL - Does not match expected <<<")

---
## Question 2: Input Generation

Validate the 4 generated input files:
- `20_03.in`: α=0.3, up to 20 nodes, 10 friends
- `20_10.in`: α=1.0, up to 20 nodes, 10 friends
- `40_03.in`: α=0.3, up to 40 nodes, 20 friends
- `40_10.in`: α=1.0, up to 40 nodes, 20 friends

In [ ]:
print("=" * 60)
print("QUESTION 2: INPUT VALIDATION")
print("=" * 60)

generated_inputs = ['20_03.in', '20_10.in', '40_03.in', '40_10.in']

for fname in generated_inputs:
    fpath = os.path.join('inputs', fname)
    if os.path.exists(fpath):
        valid, msg = is_valid_input(fpath)
        G, H, alpha = input_file_to_instance(fpath)
        status = "VALID" if valid else "INVALID"
        print(f"\n{fname}: {status}")
        print(f"  Nodes: {G.number_of_nodes()}, Friends: {len(H)}, Alpha: {alpha}")
        if not valid:
            print(f"  Issues: {msg}")
    else:
        print(f"\n{fname}: FILE NOT FOUND")

---
## Question 4.1: M-TSP Solver (Held-Karp DP)

Test the dynamic programming TSP solver on a small complete graph.

In [ ]:
print("=" * 60)
print("QUESTION 4.1: M-TSP SOLVER (HELD-KARP DP)")
print("=" * 60)

# Create a small complete graph for testing
test_graph = nx.DiGraph()
# 4-node complete graph with distances
edges = [
    (0, 1, 10), (1, 0, 10),
    (0, 2, 15), (2, 0, 15),
    (0, 3, 20), (3, 0, 20),
    (1, 2, 35), (2, 1, 35),
    (1, 3, 25), (3, 1, 25),
    (2, 3, 30), (3, 2, 30),
]
test_graph.add_weighted_edges_from(edges)

print("\nTest Graph: 4-node complete graph")
print("Edges: 0-1(10), 0-2(15), 0-3(20), 1-2(35), 1-3(25), 2-3(30)")

# Solve TSP
tsp_tour = mtsp_dp(test_graph)
print(f"\nTSP Tour: {tsp_tour}")

# Calculate tour cost
tour_cost = sum(test_graph[tsp_tour[i]][tsp_tour[i+1]]['weight'] for i in range(len(tsp_tour)-1))
print(f"Tour Cost: {tour_cost}")

# Optimal tour is 0->1->3->2->0 or 0->2->3->1->0 with cost 80
print(f"\nExpected optimal cost: 80 (0->1->3->2->0 = 10+25+30+15)")
if tour_cost == 80:
    print(">>> PASS - M-TSP Solver working correctly! <<<")
else:
    print(f">>> Got {tour_cost}, checking if valid tour... <<<")

---
## Question 4.2: PHP Solver (Reduction to TSP)

Test the PHP solver which reduces the problem to TSP.

In [ ]:
print("=" * 60)
print("QUESTION 4.2: PHP SOLVER (REDUCTION TO TSP)")
print("=" * 60)

# Test on input 1.in
G, H, alpha = input_file_to_instance('inputs/1.in')
print(f"\nTest Instance: inputs/1.in")
print(f"Nodes: {G.number_of_nodes()}, Homes: {H}, Alpha: {alpha}")

# Solve PHP
php_tour = php_solver_from_tsp(G, H)
print(f"\nPHP Tour: {php_tour}")

# Validate solution (PHP has no pickups, everyone at home)
is_valid, drive_cost, walk_cost = analyze_solution(G, H, alpha, php_tour, {})
print(f"Valid: {is_valid}")
print(f"Driving Cost: {drive_cost:.4f}")
print(f"Walking Cost: {walk_cost}")
print(f"Total Cost: {drive_cost + walk_cost:.4f}")

if is_valid:
    print(f"\n>>> PASS - PHP Solver working correctly! <<<")
else:
    print(f"\n>>> FAIL - PHP Solver has issues! <<<")

---
## Question 3: PTP Solver (Insert/Delete Heuristic)

Test the full PTP solver with pickup location optimization.

In [ ]:
print("=" * 60)
print("QUESTION 3: PTP SOLVER (INSERT/DELETE HEURISTIC)")
print("=" * 60)

# Test on input 1.in
G, H, alpha = input_file_to_instance('inputs/1.in')
print(f"\nTest Instance: inputs/1.in")
print(f"Nodes: {G.number_of_nodes()}, Homes: {H}, Alpha: {alpha}")

# Solve PTP
ptp_tour, pickups = ptp_solver(G, H, alpha)
print(f"\nPTP Tour: {ptp_tour}")
print(f"Pickup Locations: {pickups}")

# Validate solution
is_valid, drive_cost, walk_cost = analyze_solution(G, H, alpha, ptp_tour, pickups)
print(f"\nValid: {is_valid}")
print(f"Driving Cost: {drive_cost:.4f}")
print(f"Walking Cost: {walk_cost}")
print(f"Total Cost: {drive_cost + walk_cost:.4f}")

if is_valid:
    print(f"\n>>> PASS - PTP Solver working correctly! <<<")
else:
    print(f"\n>>> FAIL - PTP Solver has issues! <<<")

---
## Comparison: PHP vs PTP

Compare the two approaches on the same instance.

In [ ]:
print("=" * 60)
print("COMPARISON: PHP vs PTP SOLVERS")
print("=" * 60)

G, H, alpha = input_file_to_instance('inputs/1.in')

# PHP Solution
php_tour = php_solver_from_tsp(G, H)
_, php_drive, php_walk = analyze_solution(G, H, alpha, php_tour, {})
php_total = php_drive + php_walk

# PTP Solution
ptp_tour, pickups = ptp_solver(G, H, alpha)
_, ptp_drive, ptp_walk = analyze_solution(G, H, alpha, ptp_tour, pickups)
ptp_total = ptp_drive + ptp_walk

print(f"\n{'Method':<15} {'Driving':<12} {'Walking':<12} {'Total':<12}")
print("-" * 51)
print(f"{'PHP Solver':<15} {php_drive:<12.4f} {php_walk:<12.4f} {php_total:<12.4f}")
print(f"{'PTP Solver':<15} {ptp_drive:<12.4f} {ptp_walk:<12.4f} {ptp_total:<12.4f}")
print("-" * 51)

improvement = ((php_total - ptp_total) / php_total) * 100
print(f"\nPTP improves over PHP by {improvement:.2f}%")

---
## Test All Input Files

Run PTP solver on all available input files and show results.

In [ ]:
print("=" * 60)
print("TEST ALL INPUT FILES")
print("=" * 60)

input_files = sorted([f for f in os.listdir('inputs') if f.endswith('.in')])

print(f"\n{'File':<15} {'Nodes':<8} {'Friends':<10} {'Alpha':<8} {'Cost':<12} {'Status'}")
print("-" * 70)

for fname in input_files:
    fpath = os.path.join('inputs', fname)
    try:
        G, H, alpha = input_file_to_instance(fpath)
        tour, pickups = ptp_solver(G, H, alpha)
        is_valid, drive, walk = analyze_solution(G, H, alpha, tour, pickups)
        total = drive + walk
        status = "PASS" if is_valid else "FAIL"
        print(f"{fname:<15} {G.number_of_nodes():<8} {len(H):<10} {alpha:<8.2f} {total:<12.2f} {status}")
    except Exception as e:
        print(f"{fname:<15} ERROR: {e}")

print("\n" + "=" * 60)
print("ALL TESTS COMPLETED!")
print("=" * 60)